# New Sythnetic Diarization

In [10]:
from glob import glob
from nanodrz.constants import CACHE_DIR
from dataclasses import dataclass, field
import torchaudio
import torch
import random
from os import path

paths = glob(path.join(CACHE_DIR, "utts", "*"))


@dataclass
class Utterance:
    file: str
    speaker: str
    segements: list[tuple[int, int]] = field(default_factory=lambda: [])

    @property
    def length(self):
        return self.segements[-1][-1]

In [11]:
utts = [torch.load(p) for p in paths]
snames = list(set([u.speaker for u in utts]))
speakers = {s: [] for s in snames}

for u in utts:
    if u.speaker not in speakers:
        speakers[u.speaker] = []
    speakers[u.speaker].append(u)


for s in speakers.keys():
    utts: list[Utterance] = speakers[s]
    utts.sort(key=lambda x: x.segements[-1][-1])
    speakers[s] = utts

speakers.keys()

dict_keys(['234', '80', '507', '8956', '281', '818', '4780', '3761', '4486', '6754', '3712', '4018', '252', '1100', '1497', '1945', '439', '4217', '2234', '6788', '2909', '126', '2262', '1707', '2378', '3722', '2923', '1889', '1060', '2340', '1046', '1066', '3686', '1594', '4289', '4402', '7148', '2911', '3032', '111', '1295', '1212', '4155', '1685', '3609', '4959', '658', '4025', '5230', '5622', '1214', '1259', '7171', '2612', '5163', '2591', '1894', '3979', '2582', '1268', '11528', '4532', '5012', '2272', '4823', '1433', '3835', '1868', '4172', '3389', '1323', '5303', '551', '530', '36', '2128', '1977', '1801', '5076', '371', '2786', '979', '4214', '4411', '5809', '535', '3864', '543', '2680', '2391', '1160', '1291', '3521', '1996', '2341', '2335', '20', '2405', '263', '4837', '3727', '3723', '1463', '5129', '5259', '1251', '1834', '4312', '3028', '424', '1649', '1905', '2185', '64', '5007', '2326', '4321', '3557', '7733', '4295', '2051', '2784', '261', '4218', '3083', '1361', '2823'

In [12]:
lens = [
    sum([u.segements[-1][-1] / 16000 for u in speakers[s]]) / len(speakers[s])
    for s in speakers.keys()
]

In [13]:
def artificial_diarisation_sample(
    speakers: dict[str, Utterance] = None,
    max_secs=30,
    min_secs=7.5,
    interrupt_max=0.2,
    silence_max=0.2,
    num_speakers=4,
    sr=16000,
    **kwargs,
):
    audio = torch.zeros(1, 0)
    names, labels = [], []

    cur_speakers = random.sample(speakers.keys(), k=random.randint(2, num_speakers))
    seconds = random.uniform(min_secs, max_secs - 1)

    last_speaker = None

    for i in range(20):
        # Pick a random speaker
        speaker: str = random.choice(cur_speakers)

        cur_len = audio.shape[-1] / sr

        if seconds - cur_len < 1:
            break

        if speaker == last_speaker:
            continue

        # Determine the pause / interrupt length
        intpad = random.uniform(interrupt_max, silence_max)

        # We might not have a sample of this length or
        max_sample_len = seconds - cur_len + intpad

        max_sample_len = max(max_sample_len, 1)

        utts = speakers[speaker]
        max_sample_len = min(max([x.length for x in utts]) / sr, max_sample_len)

        # What is our smallest segment potential
        sample_len = random.uniform(0.5, max_sample_len)

        # Pick a utterance that is as long as this or longer
        utts = list(filter(lambda x: x.length > sample_len, speakers[speaker]))
        utt: Utterance = random.choice(utts)

        # Choose a random start point that gives space for the length
        # pick a segment starting with speech
        starts, ends = list(zip(*utt.segements))
        starts = [s for s in starts if s < utt.length - max_sample_len]
        start = random.choice(starts)

        # Get the best end
        end = [e < (start + sample_len) for e in ends][-1]

        sample = torchaudio.load(
            utt.file, frame_offset=int(start * sr), num_frames=int((end - start) * sr)
        )[0]

        padding = torch.zeros(1, sample.shape[-1] + intpad * sr)
        audio = torch.cat((audio, padding), dim=-1)
        audio[:, -sample.shape[-1] :] += sample

        # Derive the labels from the segment labels
        nlabels = [s for s in utt.segements if s[0] >= start and s[1] <= end]


        if speaker.name not in names:
            i = len(names)
            names.append(speaker.name)
        else:
            i = names.index(speaker.name)

        name_label = chr(ord("A") + i)
        labels += [[s[0] / sr, s[1] / sr, name_label] for s in nlabels]
        last_speaker = speaker

    return audio, labels

artificial_diarisation_sample(speakers)

/tmp/ipykernel_120721/1367007458.py:14: DeprecationWarning: Sampling from a set deprecated
since Python 3.9 and will be removed in a subsequent version.
  cur_speakers = random.sample(speakers.keys(), k=random.randint(2, num_speakers))


RuntimeError: Failed to open the input "/home/harry/.cache/nanodrz/medium/medium/2051/erewhon_0902_librivox_64kb_mp3/erewhon_15_butler_64kb_d.wav" (No such file or directory).
Exception raised from get_input_format_context at /__w/audio/audio/pytorch/audio/src/libtorio/ffmpeg/stream_reader/stream_reader.cpp:42 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::string) + 0x57 (0x7fe4e9b81d87 in /home/harry/nanodrz/.venv/lib/python3.10/site-packages/torch/lib/libc10.so)
frame #1: c10::detail::torchCheckFail(char const*, char const*, unsigned int, std::string const&) + 0x64 (0x7fe4e9b3275f in /home/harry/nanodrz/.venv/lib/python3.10/site-packages/torch/lib/libc10.so)
frame #2: <unknown function> + 0x42904 (0x7fe4d7eca904 in /home/harry/nanodrz/.venv/lib/python3.10/site-packages/torio/lib/libtorio_ffmpeg4.so)
frame #3: torio::io::StreamingMediaDecoder::StreamingMediaDecoder(std::string const&, std::optional<std::string> const&, std::optional<std::map<std::strin